## Sistema Generalizado de Cálculo de Costos de Recetas

### Arquitectura del Sistema

Este notebook implementa un sistema completamente automatizado para calcular costos de recetas basándose en precios reales de productos del supermercado.

---

### 1. Catálogo de Productos Normalizado

El sistema toma la tabla Silver `supermercadoetl_prod.silver.precios_productos` y normaliza todos los productos automáticamente:

| Columna Original | Transformación | Resultado |
|-----------------|----------------|------------|
| `unidad` | Clasificación automática | kg/medio_kg/gr/ltr/medio_ltr/ml/unidad |
| `precio_normalizado` | Sin cambios | Precio del producto |
| `descripcion` | Extracción de cantidades | Para obtener cantidad_base |

#### Reglas de Normalización Automática:

**Peso:**
* `kg` → cantidad_base = 1000 (gramos)
* `medio_kg` → cantidad_base = 500 (gramos)
* `gr` → extrae cantidad de descripción

**Volumen:**
* `ltr` → cantidad_base = 1000 (mililitros)
* `medio_ltr` → cantidad_base = 500 (mililitros)
* `ml` → extrae cantidad de descripción (maneja "2250ml" y "2.25 Lt")

**Unidades:**
* `unidad` → cantidad_base = 1
* **Caso especial - Múltiples sub-unidades**: Si la descripción contiene "X 30u" (maple de huevos, packs, etc.), extrae el número de sub-unidades
  * Ejemplo: "Huevo Blanco Maple X 30u" → cantidad_base = 30 huevos

#### Resultado: Precio Unitario
```sql
precio_unitario = precio_normalizado / cantidad_base
```

**Ejemplos:**
* Harina 1kg a $1319 → $1.319/gramo
* Aceite 500ml a $12399 → $24.798/ml
* Maple 30 huevos a $7500 → $250/huevo
* Ajo (unidad simple) a $999 → $999/unidad

---

### 2. Definición de Recetas

Las recetas se definen con nombres **simplificados** que matchean con la columna `nombre` de la tabla:

```sql
recetas_definicion AS (
  SELECT * FROM VALUES
    ('Pizza', 'Harina Trigo 0000', 500, 'gramos', 2),
    ('Pizza', 'Queso Mozzarella', 600, 'gramos', 2),
    ('Pizza', 'Ajo', 0.167, 'unidades', 2),
    ('Empanadas', 'Maple Huevo', 5, 'unidades', 3)  -- 5 huevos individuales
  AS t(receta, ingrediente, cantidad_necesaria, unidad_receta, rendimiento_porciones)
)
```

**Campos:**
* `receta`: Nombre de la receta
* `ingrediente`: Nombre simplificado (debe existir en tabla)
* `cantidad_necesaria`: Cantidad a usar (en unidades base: gramos, ml, o unidades individuales)
* `unidad_receta`: gramos/ml/unidades
* `rendimiento_porciones`: Cuántas porciones produce

**Importante**: Para productos con múltiples sub-unidades (como maples), especifica la cantidad en unidades individuales, no en packs. El sistema automáticamente detecta el pack y normaliza el precio.

---

### 3. Matching Inteligente de Productos

El sistema usa un algoritmo de matching con priorización:

**Scores de Match:**
1. **Match exacto** (score=1): "Cebolla" = "Cebolla"
2. **Empieza con** (score=2): "Cebolla" en "Cebolla caramelizada"
3. **Contiene** (score=3): "Cebolla" en "Sopa de Cebolla"

**Desempate:** Si hay múltiples matches con el mismo score, elige el nombre más corto (más específico).

Esto evita matcheos incorrectos como "Cebolla" con "Cebolla de verdeo".

---

### 4. Cálculo de Costos

```sql
costo_ingrediente = precio_unitario × cantidad_necesaria
costo_total_receta = SUM(costo_ingrediente)
costo_por_porcion = costo_total_receta / rendimiento_porciones
```

**Ejemplo Real:**

**Pizza (rinde 2 pizzas):**
* Harina 500g: $1.319 × 500 = $659.50
* Queso 600g: $17.275 × 600 = $10,365.00
* Ajo 0.167u: $999 × 0.167 = $166.83
* **Total**: $13,192.82 → **$6,596.41/pizza**

**Empanadas (rinde 3 docenas):**
* Maple Huevo (5 huevos): $250 × 5 = $1,250.00
* ... (otros ingredientes)
* **Total**: $29,064.47 → **$9,688.16/docena**

---

### 5. Ventajas del Sistema

✅ **100% Automatizado**: No requiere hardcodeo de productos  
✅ **Escalable**: Agregar recetas = agregar filas a VALUES  
✅ **Mantenible**: Precios se actualizan automáticamente  
✅ **Robusto**: Matching inteligente evita errores  
✅ **Flexible**: Soporta peso, volumen, unidades simples y packs  
✅ **Auditable**: Query de detalle muestra cada ingrediente

---

### 6. Cómo Agregar Nuevas Recetas

**Paso 1:** Busca los nombres simplificados en la tabla:
```sql
SELECT DISTINCT nombre, descripcion, unidad
FROM supermercadoetl_prod.silver.precios_productos
WHERE LOWER(nombre) LIKE '%ingrediente%'
```

**Paso 2:** Agrega las filas a `recetas_definicion` con los nombres exactos

**Paso 3:** Ejecuta la celda - el sistema calcula todo automáticamente

---

### 7. Query de Detalle

Para auditar ingrediente por ingrediente, descomenta la última línea de la celda 4:

```sql
SELECT * FROM calculo_costos 
WHERE fecha_extraccion >= '2026-04-15'
ORDER BY receta, ingrediente
```

Muestra: producto matcheado, descripción, precio, cantidad_base, precio_unitario, cantidad necesaria, costo calculado.

In [0]:
%sql
-- ============================================
-- SISTEMA GENERALIZADO DE CÁLCULO DE RECETAS
-- ============================================

-- 1. CATÁLOGO DE PRODUCTOS NORMALIZADO
With catalogo_productos AS(
  SELECT 
  nombre,
  descripcion,
  categoria,
  precio_normalizado,
  unidad,
  CAST(CASE 
        WHEN unidad IN ('kg','ltr') THEN 1000
        WHEN unidad IN ('medio_ltr','medio_kg') THEN 500
        -- Manejo especial para productos con múltiples sub-unidades (ej: Maple X 30u)
        WHEN unidad = 'unidad' AND (descripcion LIKE '%X %u%' OR descripcion LIKE '%x %u%') THEN
          CAST(NULLIF(REGEXP_EXTRACT(TRIM(descripcion), '[Xx] (\\d+)u', 1),'') AS INT)
        WHEN unidad = 'unidad' THEN 1
        -- Manejo especial para ml: extraer de descripción
        WHEN unidad = 'ml' THEN
          CASE
            -- Si tiene "ml" en descripción, extraer el número (ej: 2250ml)
            WHEN descripcion LIKE '%ml%' THEN CAST(NULLIF(REGEXP_EXTRACT(TRIM(descripcion), '(\\d+)ml', 1),'') AS INT)
            -- Si tiene "Lt" o "L" en descripción, extraer y convertir a ml (ej: 2,25 Lt -> 2250)
            WHEN descripcion LIKE '%Lt%' OR descripcion LIKE '%L %' OR descripcion LIKE '% L' THEN 
              CAST(CAST(REPLACE(NULLIF(REGEXP_EXTRACT(TRIM(descripcion), '([0-9]+[.,]?[0-9]*)', 1),''), ',', '.') AS DOUBLE) * 1000 AS INT)
            ELSE 1
          END
        -- Para otras unidades con gr, extraer el número de la descripción
        ELSE CAST(NULLIF(REGEXP_EXTRACT(TRIM(descripcion), '(\\d+)', 0),'') AS INT)
    END AS INT) As cantidad_base, 
  fecha_extraccion,
  url
  FROM supermercadoetl_prod.silver.precios_productos
),
catalogo_normalizado AS (
  SELECT
  nombre,
  descripcion,
  categoria,
  precio_normalizado,
  unidad,
  cantidad_base,
  ROUND(precio_normalizado/cantidad_base,4) AS precio_unitario,
  fecha_extraccion,
  url
  FROM catalogo_productos
),

-- 2. DEFINICIÓN DE RECETAS
recetas_definicion AS (
  SELECT * FROM VALUES
    -- Pizza (rinde 2 pizzas)
    ('Pizza', 'Harina Trigo 0000', 500, 'gramos', 2),
    ('Pizza', 'Condimento Para Pizza', 3, 'gramos', 2),
    ('Pizza', 'Sal Fina', 10, 'gramos', 2),
    ('Pizza', 'Pimenton', 2, 'gramos', 2),
    ('Pizza', 'Orégano', 5, 'gramos', 2),
    ('Pizza', 'Tomate Perita en lata', 1, 'unidades', 2),
    ('Pizza', 'Levadura Seca', 1, 'unidades', 2),
    ('Pizza', 'Aji Molido', 2, 'gramos', 2),
    ('Pizza', 'Ajo', 0.167, 'unidades', 2), -- 1/6 de ajo
    ('Pizza', 'Queso Mozzarella', 600, 'gramos', 2),
    -- Empanadas (rinde 3 docenas = 36 empanadas)
    ('Empanadas', 'Picada Desgrasada', 1000, 'gramos', 3),
    ('Empanadas', 'Tapa Para Empanadas X12', 3, 'unidades', 3),
    ('Empanadas', 'Cebolla', 200, 'gramos', 3),
    ('Empanadas', 'Puerro', 2, 'unidades', 3),
    ('Empanadas', 'Cebolla de verdeo', 2, 'unidades', 3),
    ('Empanadas', 'Zanahoria', 150, 'gramos', 3),
    ('Empanadas', 'Morron rojo', 250, 'gramos', 3),
    ('Empanadas', 'Sal Fina', 50, 'gramos', 3),
    ('Empanadas', 'Condimento Para Empanadas', 5, 'gramos', 3),
    ('Empanadas', 'Aceitunas Verdes Descarozadas', 1, 'unidades', 3),
    ('Empanadas', 'Maple Huevo', 5, 'unidades', 3)
  AS t(receta, ingrediente, cantidad_necesaria, unidad_receta, rendimiento_porciones)
),

-- 3. CÁLCULO DE COSTOS POR INGREDIENTE (con match inteligente)
calculo_costos AS (
  SELECT 
    r.receta,
    r.ingrediente,
    c.nombre AS nombre_producto_completo,
    c.descripcion,
    c.categoria,
    c.precio_normalizado AS precio_producto,
    c.cantidad_base,
    c.precio_unitario,
    c.unidad,
    r.cantidad_necesaria,
    r.unidad_receta,
    -- Cálculo del costo del ingrediente
    ROUND(c.precio_unitario * r.cantidad_necesaria, 2) AS costo_ingrediente,
    r.rendimiento_porciones,
    c.fecha_extraccion,
    -- Score de match para priorizar el mejor
    CASE 
      WHEN TRIM(LOWER(c.nombre)) = TRIM(LOWER(r.ingrediente)) THEN 1  -- Match exacto
      WHEN TRIM(LOWER(c.nombre)) LIKE CONCAT(TRIM(LOWER(r.ingrediente)), '%') THEN 2  -- Empieza con
      ELSE 3  -- Contiene
    END AS match_score,
    LENGTH(c.nombre) as nombre_length
  FROM catalogo_normalizado c
  INNER JOIN recetas_definicion r
    ON TRIM(LOWER(c.nombre)) LIKE CONCAT('%', TRIM(LOWER(r.ingrediente)), '%')
  -- Priorizar: (1) mejor match_score, (2) nombre más corto (más específico)
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY r.receta, r.ingrediente, c.fecha_extraccion 
    ORDER BY match_score ASC, nombre_length ASC
  ) = 1
)

-- 4. RESUMEN FINAL POR RECETA
SELECT 
  receta,
  fecha_extraccion,
  ROUND(SUM(costo_ingrediente), 2) AS costo_total,
  MAX(rendimiento_porciones) AS porciones,
  ROUND(SUM(costo_ingrediente) / MAX(rendimiento_porciones), 2) AS costo_por_porcion,
  CASE 
    WHEN receta = 'Pizza' THEN 'Porcion = 1 pizza'
    WHEN receta = 'Empanadas' THEN 'Porcion = 1 docena'
    ELSE NULL
  END AS comentarios
FROM calculo_costos
WHERE fecha_extraccion >= '2026-04-15'
GROUP BY receta, fecha_extraccion
ORDER BY receta, fecha_extraccion

-- Para ver detalle por ingrediente, descomenta:
-- SELECT * FROM calculo_costos 
-- WHERE fecha_extraccion >= '2026-04-15'
-- ORDER BY receta, ingrediente, fecha_extraccion

## Cómo Agregar Más Recetas al Sistema

### El Sistema es Completamente Generalizado

**No necesitas modificar el código del catálogo** - trabaja con TODOS los productos de la tabla Silver automáticamente.

---

### Paso 1: Buscar Nombres de Productos Disponibles

```sql
-- Ver todos los productos disponibles
SELECT DISTINCT nombre, unidad, precio_normalizado, descripcion
FROM supermercadoetl_prod.silver.precios_productos
WHERE LOWER(nombre) LIKE '%ingrediente_que_busco%'
ORDER BY nombre
```

**Ejemplos de búsquedas:**
* `WHERE LOWER(nombre) LIKE '%leche%'` → Ver productos lácteos
* `WHERE LOWER(nombre) LIKE '%aceite%'` → Ver aceites
* `WHERE categoria = 'Carnes'` → Ver todas las carnes

---

### Paso 2: Agregar Ingredientes a tu Receta

Simplemente agrega filas al CTE `recetas_definicion` usando los nombres **exactos** de la columna `nombre`:

```sql
recetas_definicion AS (
  SELECT * FROM VALUES
    -- Recetas existentes...
    
    -- Nueva receta: Salsa Bechamel (rinde 4 porciones)
    ('Salsa Bechamel', 'Leche Entera', 500, 'ml', 4),
    ('Salsa Bechamel', 'Harina Trigo 0000', 50, 'gramos', 4),
    ('Salsa Bechamel', 'Manteca', 50, 'gramos', 4),
    ('Salsa Bechamel', 'Sal Fina', 5, 'gramos', 4),
    
    -- Nueva receta: Tarta de verduras (rinde 8 porciones)
    ('Tarta', 'Masa Para Tarta', 2, 'unidades', 8),
    ('Tarta', 'Acelga', 500, 'gramos', 8),
    ('Tarta', 'Cebolla', 200, 'gramos', 8),
    ('Tarta', 'Queso Crema', 200, 'gramos', 8),
    ('Tarta', 'Maple Huevo', 3, 'unidades', 8)
    
  AS t(receta, ingrediente, cantidad_necesaria, unidad_receta, rendimiento_porciones)
)
```

---

### Paso 3: El Sistema Hace Todo Automáticamente

✅ **Busca el producto** con matching inteligente  
✅ **Identifica la unidad** (kg/gr/ltr/ml/unidad) automáticamente  
✅ **Extrae cantidad_base** de la descripción  
✅ **Calcula precio_unitario** ($/gramo, $/ml, $/unidad)  
✅ **Multiplica** precio_unitario × cantidad_necesaria  
✅ **Suma todos los ingredientes** y divide por rendimiento

---

### Manejo de Diferentes Tipos de Productos

El sistema soporta **3 tipos de medida** automáticamente:

**1. Productos de Peso (gramos)**
```sql
('Receta', 'Harina Trigo 0000', 500, 'gramos', 4)  -- Usa kg/medio_kg/gr
('Receta', 'Queso Mozzarella', 300, 'gramos', 4)
```

**2. Productos de Volumen (mililitros)**
```sql
('Receta', 'Leche Entera', 250, 'ml', 4)  -- Usa ltr/medio_ltr/ml
('Receta', 'Aceite Oliva', 100, 'ml', 4)
```

**3. Productos Unitarios**
```sql
('Receta', 'Maple Huevo', 3, 'unidades', 4)  -- Usa unidad
('Receta', 'Tapa Para Empanadas X12', 2, 'unidades', 4)
('Receta', 'Ajo', 0.5, 'unidades', 4)  -- Puede ser fraccional
```

---

### Ejemplo Completo: Agregar Receta de Milanesas

```sql
recetas_definicion AS (
  SELECT * FROM VALUES
    -- ... recetas existentes ...
    
    -- Milanesas (rinde 4 porciones)
    ('Milanesas', 'Carne para Milanesa', 800, 'gramos', 4),
    ('Milanesas', 'Pan Rallado', 200, 'gramos', 4),
    ('Milanesas', 'Maple Huevo', 2, 'unidades', 4),
    ('Milanesas', 'Sal Fina', 10, 'gramos', 4),
    ('Milanesas', 'Aceite de Girasol', 100, 'ml', 4)
    
  AS t(receta, ingrediente, cantidad_necesaria, unidad_receta, rendimiento_porciones)
)
```

**Resultado automático:**
* Costo total de la receta
* Costo por porción (divide por 4)
* Detalle de cada ingrediente

---

### Productivizar: Crear Tablas Gold

Para usar esto en producción, considera crear tablas permanentes:

```sql
-- Catálogo normalizado (actualizar diariamente)
CREATE OR REPLACE TABLE supermercadoetl_prod.gold.catalogo_productos_normalizado AS
SELECT 
  nombre,
  precio_normalizado,
  unidad,
  cantidad_base,
  precio_unitario,
  fecha_extraccion
FROM catalogo_normalizado;

-- Recetas (mantener manualmente)
CREATE OR REPLACE TABLE supermercadoetl_prod.gold.recetas_definicion AS
SELECT * FROM recetas_definicion;

-- Costos históricos (actualizar diariamente)
CREATE OR REPLACE TABLE supermercadoetl_prod.gold.costos_recetas AS
SELECT 
  receta,
  fecha_extraccion,
  costo_total,
  porciones,
  costo_por_porcion
FROM calculo_costos;
```

Luego puedes crear dashboards, alertas de precios, análisis de tendencias, etc.

In [0]:
%sql
-- 4. RESUMEN FINAL POR RECETA
SELECT 
  receta,
  fecha_extraccion,
  ROUND(SUM(costo_ingrediente), 2) AS costo_total,
  MAX(rendimiento_porciones) AS porciones,
  ROUND(SUM(costo_ingrediente) / MAX(rendimiento_porciones), 2) AS costo_por_porcion,
  CASE 
    WHEN receta = 'Pizza' THEN 'Porcion = 1 pizza'
    WHEN receta = 'Empanadas' THEN 'Porcion = 1 docena'
    ELSE NULL
  END AS comentarios
FROM supermercadoetl_prod.gold.costos_recetas
WHERE fecha_extraccion >= '2026-04-15'
GROUP BY receta, fecha_extraccion
ORDER BY receta, fecha_extraccion